# L3b: Lattice Models for Equity Pricing
Today, we will introduce lattice models of equity share prices. We’ll estimate the lattice model parameters using a data-driven approach and introduce a model-driven approach today that we will develop in a later lecture. Finally, we explore whether a lattice model shows any of the stylized facts we find in market observations. 

> __Learning Objectives:__
>
> By the end of this lecture, you will be able to define and demonstrate mastery of the following key concepts:
> * __Binomial lattice model:__ The model assumes that tomorrow exists in one of two possible states: up or down. From today, we move to the up-state with probability p and the down-state with probability (1 - p).
> * __Lattice model parameters:__ The lattice parameters can be set two ways: estimated from historical data under real-world probabilities, or fixed by a no-arbitrage condition under risk-neutral probabilities. The first law supports forecasting and the second supports valuation, and the two are not interchangeable.
> * __Stylized facts:__ Stylized facts are recurring statistical properties of asset returns across markets, including heavy tails, volatility clustering, and the absence of linear autocorrelation. They serve as tests a candidate price model must pass before it is trusted. Does a lattice model show any of the stylized facts we observe in market data? We will explore this question today.

While lattice models are not the most sophisticated models available, they are easy to understand and implement, and they provide a workable first model of share-price uncertainty over short time periods. Let's get started!
___

## Examples
Today, we will be using the following example(s) to illustrate key concepts:

* [▶ Binomial Lattice Simulation of Equity Share Price using Real-World Probability](CHEME-5660-L3b-LatticeModelSharePrice-RWPM-Example-Fall-2026.ipynb). In this example, we will develop a binomial lattice model of equity share prices using a real-world probability approach. We will simulate the share price over a specified time horizon, and then analyze the results to see if the model captures any of the stylized facts we observe in real market data.

Optional advanced notebooks that extend today's material are listed in the Optional Advanced Material section at the end of this lecture.
___

## Binomial Lattice Model
A binomial lattice model is a discrete-time model used to represent the evolution of an asset's price over time using a recombining tree structure. 

<div>
    <center>
        <img src="figs/Fig-Lattice-Schematic.svg" width="800" alt="Schematic of a recombining binomial lattice growing through three panels: from the current price at time zero, a one-step tree branches to an up node with probability p and factor u or a down node with probability 1 minus p and factor d; a two-step tree extends to the uu, ud, and dd nodes; and a three-step tree ends in four terminal price nodes"/>
    </center>
</div>

> __What is a binomial lattice model?__ 
> 
> A binomial lattice model assumes that at each time step, the asset can either move up by a factor $u$ with a probability $p$ or down by a factor $d$ with probability $(1-p)$, creating a tree-like structure of possible future prices. At time step $t\geq{0}$ there will be $t+1$ possible prices. 
>
> __Key assumptions__: The probability $p$, and the up and down factors are independent of the time step $t$, i.e., these parameters are constant across the lattice. The up, down and probability parameters are specific to each firm.

Let's see how this works in practice.

### Price Distribution

Suppose we have a stock with an initial price $S_0$ at time $t=0$ (today). Assuming a binomial lattice model, at $t = 1$ (tomorrow), the stock can occupy one of two possible states: the up state with a price $S^{(+)}_{1} = S_{0}\;u$ with probability $p$ or the down state with price $S^{(-)}_{1} = S_{0}\;d$ with probability $(1-p)$.  Thus, the future prices at time $t = 1$ can be expressed as:
$$
S_{1} = \begin{cases}
S^{(+)}_{1} = S_{0}\;u & \text{with probability } p \\
S^{(-)}_{1} = S_{0}\;d & \text{with probability } (1-p)
\end{cases}
$$
The process continues for $N$ time steps (the number of steps to the simulation horizon), creating a binomial tree of possible prices.


The binomial lattice model gets its name because the number of up moves after $t$ steps follows a binomial distribution, which indexes the possible prices at level $t$. Let's see where this comes from. 

> __Key idea__: At each node of the lattice, we are performing a Bernoulli trial: we either move up with probability $p$ or down with probability $(1-p)$. Thus, we have a sequence of $t$ independent Bernoulli trials, where each trial has two possible outcomes: up or down. The number of up moves in this sequence of trials follows a binomial distribution.

Consider a path from the root ($t=0$) to any node at level $t$. This path consists of exactly $t$ moves, where each move is either up or down. Let $k$ be the number of up moves and $(t-k)$ be the number of down moves. The price at any node with $k$ up moves and $(t-k)$ down moves is:
$$
S_{t,k} = S_0 \cdot u^k \cdot d^{t-k}
$$
The probability of reaching this specific node is given by the binomial probability:
$$
P(k \text{ up moves in } t \text{ steps}) = \binom{t}{k}\;p^k (1-p)^{t-k}
$$

where $\binom{t}{k} = \frac{t!}{k!(t-k)!}$ is the binomial coefficient representing the number of ways to choose $k$ up moves from $t$ total moves.
At level $t$ (e.g., $t$ days in the future), there are $(t+1)$ possible prices:
$$
\begin{align*}
S_{t,0} &= S_0 d^t &&\text{(all down moves)} \\
S_{t,1} &= S_0 u^1 d^{t-1} &&\text{(one up, } (t-1) \text{ down)} \\
&\vdots \\
S_{t,t} &= S_0 u^t &&\text{(all up moves)}
\end{align*}
$$
Thus, the complete price distribution at level $t$ follows:
$$
\boxed{
S_t = S_0 u^k d^{t-k} \quad \text{with probability} \quad \binom{t}{k}\;p^k (1-p)^{t-k}, \quad k = 0,1,\ldots,t
}
$$

This is why it's called a **binomial** lattice model - the number of up moves at each level follows a binomial distribution with parameters $t$ (number of trials) and $p$ (probability of success/up move).

So where do we get the up and down factors $u$ and $d$, and the probability $p$? There are two approaches: a data-driven approach and a model-driven approach. Let's start with the data-driven approach.

### Data-Driven Real World Parameter Estimation
For us to simulate future share price (and return/growth rate) distributions, we need to estimate the up and down factors $u$ and $d$, and the probability $p$. We can estimate these parameters $(u,d,p)$ from historical data.

Let's estimate the up $u$, down $d$ factors and the probability $p$ from historical data.

__Initialize__: Given the growth rate sequence $\{g_{2},g_{3},\dots,g_{T}\}$ for firm $(i)$ (we neglect the superscript $i$ for simplicity) and a time step $\Delta{t} > 0$ (units: years). Initialize an empty up-factor list $U$ and an empty down-factor list $D$.

1. For $t = 2,3,\dots,T$ __do__:
    - If $g_{t} > 0$, then append $e^{g_{t}\Delta{t}}$ to the up-factor collection $U$.
    - If $g_{t} < 0$, then append $e^{g_{t}\Delta{t}}$ to the down-factor collection $D$.
    - If $g_{t} = 0$, skip (no price change).
2. Compute the up factor $u$ as the mean of the up factors collection $U$: $u = \frac{1}{|U|} \sum_{v \in U} v$.
3. Compute the down factor $d$ as the mean of the down factors collection $D$: $d = \frac{1}{|D|} \sum_{v \in D} v$.
4. Compute the probability $p$ as the fraction of all observed steps that are up moves: $p = |U|/(T-1)$, where a zero-change day counts in the denominator but joins neither collection. The estimator requires at least one up move and one down move in the sample.

> __Note on recombination__: This data-driven approach produces a recombining lattice since $ud = du$ (multiplication is commutative). Later, we'll impose the constraint $ud = 1$, however, this condition is not necessary for recombination, though it may be imposed in some models for risk-neutral pricing or to ensure symmetric price movements.

Let's see this estimation procedure in action:

> __Example__
>
> [▶ Binomial Lattice Simulation of Equity Share Price using Real-World Probability](CHEME-5660-L3b-LatticeModelSharePrice-RWPM-Example-Fall-2026.ipynb). In this example, we will develop a binomial lattice model of equity share prices using a real-world probability approach. We will simulate the share price over a specified time horizon, and then analyze the results to see if the model captures any of the stylized facts we observe in real market data.


### Model-Driven Risk-Neutral Probability
The data-driven method estimates the up factor $u$, the down factor $d$, and the real-world up probability $p$ from historical returns. It defines the forecasting rule $\mathbb{P}$, where $\mathbb{P}(\text{up})=p$ and $\mathbb{P}(\text{down})=1-p$. The model-driven method keeps $u$ and $d$ but replaces $p$ with the risk-neutral up probability $q$ to define the risk-neutral pricing rule $\mathbb{Q}$, where $\mathbb{Q}(\text{up})=q$ and $\mathbb{Q}(\text{down})=1-q$. Risk-neutral probabilities are examples of arbitrage-free probabilities. 

An __arbitrage__ is a zero-cost trade that cannot lose money and it makes money in at least one future state. Let's assume we have a risk-free investment with a continuously compounded growth rate $g_y$ (units: $1/\text{year}$). Over a step of length $\Delta t$ years, the risk-free accumulation factor is $R_f=e^{g_y\Delta t}$. 

To avoid arbitrage, $d<R_f<u$:
* If $R_f\leq d$, borrow money at the risk-free nominal rate $y$ and then buy the share. The share repays the loan in both states and leaves a profit after an up move. __Example:__ Let $S_0=100$, $R_f=d=1.05$, and $u=1.20$. Borrow 100 and buy one share. Next period, sell the share for 105 after a down move or 120 after an up move, then repay 105. The profit is 0 or 15, respectively.
* If $R_f\geq u$, sell a borrowed share and invest the proceeds in the risk-free asset. The investment buys back the share in both states and leaves a profit after a down move. __Example:__ Let $S_0=100$, $R_f=u=1.10$, and $d=0.90$. Sell a borrowed share for 100 and invest the proceeds. Next period, the investment is worth 110; repurchase the share for 110 after an up move or 90 after a down move. The profit is 0 or 20, respectively.

The risk-neutral pricing rule $\mathbb{Q}$ assigns probability $q$ to an up move and $1-q$ to a down move. To avoid arbitrage, the probability-weighted average of $u$ and $d$ must equal the risk-free factor: $qu+(1-q)d=R_f$. Solving for $q$ gives:
$$
\boxed{q=\frac{R_f-d}{u-d}}.
$$
Since $d<R_f<u$, the formula gives $0<q<1$. Why is $q$ useful? Suppose a financial claim pays $H_u$ dollars next period if the share moves up and $H_d$ dollars if it moves down. Under the risk-neutral pricing rule, $qH_u+(1-q)H_d$ is the weighted next-period payoff. Dividing by $R_f$ discounts that payoff back one step, so the present value, or no-arbitrage price, of the contingent claim is:
$$
V_0=\frac{qH_u+(1-q)H_d}{R_f}.
$$
Here, $V_0$ is a risk-neutral present value under $\mathbb{Q}$; $q$ is a pricing weight, not a prediction of how often up moves occur under $\mathbb{P}$. We will derive this rule when we study options later in the course.
___

## Stylized Facts for Binomial Lattice Models
A useful synthetic growth model should reproduce the stylized facts seen in market data: does a binomial lattice model show heavy tails, little linear autocorrelation in signed growth rates, and volatility clustering?

Because stylized facts describe real-world growth rates, we evaluate the lattice under $\mathbb{P}$. The conclusions below are structural: changing the fixed up-move probability alters the distribution's numerical moments but leaves the one-step growth rates bounded and independent with constant variance.

### Growth-Rate Distribution
We first determine the one-step growth-rate distribution generated by the lattice and ask whether it can have heavy tails.

> __One-Step Growth-Rate Distribution Theorem (constant-parameter binomial lattice):__
>
> Let $S_0>0$ be the initial share price, $\Delta t>0$ be the step length in years, and $u>d>0$ be constant move factors. Under the real-world probability rule $\mathbb{P}$, the up-move probability is $p$ and the down-move probability is $1-p$, where $0<p<1$. 
> 
> For each step $j$, let $X_j=1$ for an up move and $X_j=0$ for a down move. Assume the indicators $X_1,X_2,\ldots$ are independent. The one-step price factor is then $S_j/S_{j-1}=u^{X_j}d^{1-X_j}$. The corresponding one-step continuously compounded growth rate can then be written as:
> $$
> g_j\equiv g_{j,j-1}=\frac{1}{\Delta t}\ln\left(\frac{S_j}{S_{j-1}}\right)
> =\frac{\ln d+X_j\ln(u/d)}{\Delta t}.
> $$
> where $g_j$ has units of inverse years. The mean $\mu_g$ and variance $\sigma_g^2$ of the growth rates under the real-world probability rule $\mathbb{P}$ are given by:
> $$
> \begin{aligned}
> \mu_g\equiv\mathbb{E}_{\mathbb{P}}[g_j]
> &=\frac{p\ln u+(1-p)\ln d}{\Delta t},\\
> \sigma_g^2\equiv\operatorname{Var}_{\mathbb{P}}(g_j)
> &=\frac{p(1-p)}{(\Delta t)^2}\left[\ln\left(\frac{u}{d}\right)\right]^2.
> \end{aligned}
> $$
> The growth rate therefore satisfies:
> $$
> g_j\in\left\{\frac{\ln d}{\Delta t},\frac{\ln u}{\Delta t}\right\},\qquad
> \lvert g_j\rvert\leq M\equiv\frac{\max\{\lvert\ln d\rvert,\lvert\ln u\rvert\}}{\Delta t}.
> $$
> Thus, $g_j$ has a two-point discrete distribution and, crucially, is bounded. For any threshold $x>M$, $\mathbb{P}(\lvert g_j\rvert>x)=0$. Changing $p$ changes the probability assigned to each value but not this finite bound. The zero tail probability beyond $M$ is why the model cannot generate heavy-tailed one-step growth rates; discreteness by itself is not the deciding property.

The constant-parameter lattice therefore does not reproduce the heavy tails observed in market growth rates. The cumulative log-return distribution is useful for longer-horizon questions, but it is not needed for this one-step stylized-fact test; its derivation appears in the optional notebook below.


### Temporal Dependence
The one-step growth-rate distribution theorem answers the heavy-tail question. However, it does not address the question of temporal dependence. The remaining two stylized facts concern dependence between growth rates at different time steps.

> __Temporal Dependence Theorem (constant-parameter binomial lattice):__
>
> Keep the assumptions of the one-step growth-rate distribution theorem, and let $\tau\geq1$ be an integer lag. Because $g_j$ is a function of $X_j$, independence of the move indicators gives $\operatorname{Cov}_{\mathbb{P}}(g_j,g_{j+\tau})=0$. Since $0<p<1$ and $u\neq d$, the variance $\sigma_g^2$ is positive. Thus, the one-step growth-rate autocorrelation is given by:
> $$
> \rho_g(\tau)\equiv
> \frac{\operatorname{Cov}_{\mathbb{P}}(g_j,g_{j+\tau})}{\sigma_g^2}=0.
> $$
> Independence also applies to the growth-rate magnitudes, while the one-step variance remains constant. These two results can be written as:
> $$
> \operatorname{Cov}_{\mathbb{P}}\left(\lvert g_j\rvert,\lvert g_{j+\tau}\rvert\right)=0,\qquad
> \operatorname{Var}_{\mathbb{P}}(g_j)=\sigma_g^2\quad\text{for every }j.
> $$

Taken together, the two theorems give a precise scorecard. The constant-parameter binomial lattice reproduces zero linear autocorrelation in signed one-step growth rates. However, it does __not__ reproduce heavy tails or volatility clustering.

> [▶ Optional derivation notebook: Binomial Growth Rate and Return Derivations](advanced/stylized-facts/CHEME-5660-L3b-Advanced-BinomialReturns-Derivations-Fall-2026.ipynb). In this notebook, we derive the one-step growth-rate law and its moments, then extend the calculation to the distribution of the multi-step cumulative log return. We also derive the Normal limiting behavior, zero autocorrelation of distinct one-step growth rates, correlation caused by overlapping return windows, and the constant-variance result used above.

___

## Optional Advanced Material
The notebooks below extend today's material. They are optional and are not prerequisites for L4a. The [advanced index](advanced/README.md) lists them with a suggested order.

* [▶ Recombination and Computational Cost](advanced/recombination/CHEME-5660-L3b-Advanced-Recombination-ComputationalCost-Fall-2026.ipynb). In this optional structure notebook, we distinguish replacement from recombination and show why constant move factors recombine even when $d\neq 1/u$. We then compare the exponential node count of a full path tree with the quadratic node count of a recombined lattice and derive the two-step ratio condition that time-dependent factors must satisfy to recombine.
* [▶ Bootstrap Uncertainty in Lattice Calibration](advanced/calibration/CHEME-5660-L3b-Advanced-Bootstrap-LatticeCalibration-Fall-2026.ipynb). In this optional calibration notebook, we estimate $(u,d,p)$ from historical growth rates and use independent and circular moving-block bootstraps to measure sampling uncertainty. We propagate that uncertainty into the expected terminal price and $\mathbb{P}(S_N>S_0)$, compare interval widths under the two resampling rules, and check the fitted model against held-out observations.
* [▶ Binomial Return Derivations](advanced/stylized-facts/CHEME-5660-L3b-Advanced-BinomialReturns-Derivations-Fall-2026.ipynb). In this optional derivation notebook, we derive the one-step growth-rate law and its moments, then extend the calculation to the distribution of the multi-step cumulative log return. We also derive the Normal limiting behavior, zero autocorrelation of distinct one-step growth rates, correlation caused by overlapping return windows, and the constant-variance result used above.
___

## Summary
In this lecture, we introduced the binomial lattice model for equity share prices. We discussed how to estimate the model parameters using a data-driven approach. Finally, we explored whether the binomial lattice model captures any of the stylized facts observed in real market data.

> __Key Takeaways:__
>
> * __Binomial lattice model:__ We modeled the next price as one of two outcomes, reached from the current state with a fixed probability and its complement. Repeating that step builds a binomial price distribution whose spread widens with the horizon.
> * __Lattice model parameters:__ We estimated the move magnitudes and the transition probability from a historical growth-rate sample, which gives the real-world law used for forecasting. The risk-neutral probability instead follows from the move factors and the risk-free return, and it is the law used for pricing.
> * __Stylized facts:__ The binomial lattice reproduces the absence of autocorrelation in one-step growth rates. It does not produce heavy tails or volatility clustering, so it fails two of the three diagnostics the previous lecture established.

While lattice models are not the most sophisticated models available, they are easy to understand and implement, and they provide a workable first model of share-price uncertainty over short horizons, even though they miss several of the stylized facts we examined.

___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance.  Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.
